<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/extraction-pipeline/Extraction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Notebook setup - drive, HF login, paths, seed

In [10]:
!pip install -q datasets huggingface_hub openpyxl

from google.colab import drive, userdata
from huggingface_hub import login, HfApi
import os

drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

BASE        = "/content/drive/MyDrive/Research-WSD-New"
OUT         = f"{BASE}/extraction_v5"
os.makedirs(OUT, exist_ok=True)

DATASET     = "sinhala-nlp/sinhala-1.5B-corpus"
REVISION    = HfApi().dataset_info(DATASET).sha
RANDOM_SEED = 42
RESERVOIR_K = 1000   # sentences kept per word per source during the scan
print("Dataset revision:", REVISION)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset revision: 352f711af658137be6acceea60063c9e711bf4a4


### 2. Load target words

In [11]:
import re, unicodedata
from collections import defaultdict
import openpyxl

WORDS_XLSX  = f"{BASE}/WSD_Polysemy_Words.xlsx"
WORDS_SHEET = "target_words"
EXPECTED_N  = 70

JOINERS = re.compile(r"[\u200C\u200D]")
def match_key(tok):
    """Key used ONLY for matching: NFC + ZWJ/ZWNJ removed, so ප්\u200Dර and ප්ර match the same."""
    return JOINERS.sub("", unicodedata.normalize("NFC", str(tok)))

def load_target_words(path=WORDS_XLSX, sheet=WORDS_SHEET):
    """-> words (sheet order), {word: No}, {word: [spellings]}."""
    rows = openpyxl.load_workbook(path, read_only=True)[sheet].iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]
    i_no, i_word = header.index("No"), header.index("Word")
    words, numbers, spellings, seen_no = [], {}, {}, set()
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        w, no = str(r[i_word]).strip(), int(float(r[i_no]))
        if w in numbers or no in seen_no:
            raise ValueError(f"Duplicate word or number in sheet: {no} {w}")
        words.append(w); numbers[w] = no; seen_no.add(no)
        spellings[w] = [s.strip() for s in re.split(r"[/,]", w) if s.strip()]
    return words, numbers, spellings

TARGETS, word_no, word_spellings = load_target_words()
assert len(TARGETS) == EXPECTED_N, f"Expected {EXPECTED_N} words, got {len(TARGETS)}"

MATCH = {}
for w in TARGETS:
    for s in word_spellings[w]:
        k = match_key(s)
        if k in MATCH and MATCH[k][0] != w:
            raise ValueError(f"Spelling {s} belongs to two words: {MATCH[k][0]} and {w}")
        MATCH[k] = (w, s)
print(f"{len(TARGETS)} words | {len(MATCH)} spellings")
print(TARGETS)

70 words | 79 spellings
['දිය', 'දද', 'නා', 'ගත', 'පස / පස්', 'මුව', 'වත / වත්', 'තෙර', 'දන', 'ලිය', 'උර', 'තල', 'නල', 'පෙළ', 'මද', 'මී', 'රැස', 'වර', 'වැට', 'අත', 'ගල', 'උත්තර', 'කඩ', 'කම්', 'කලාපය', 'සුර', 'සත්', 'සල් / සල', 'පත / පත්', 'සක / සක්', 'පැහැ', 'තුරු', 'තිර', 'සඳ', 'සිළ / සිළු', 'හස / හස්', 'පණ', 'පිය', 'දත්', 'කලි', 'පට', 'රද', 'දෙස් / දෙස', 'සමර', 'වස්', 'සර', 'කුළු', 'රජ', 'රස', 'නව', 'සතර', 'දීප', 'වැඩ', 'වණ', 'වද', 'අස්න', 'කර', 'දල', 'නිදන', 'නද', 'රා', 'ලේ', 'සී', 'කල/කල්', 'අඹ', 'කමල', 'මාන', 'සව්', 'ඇලි', 'තිත්']


### 3. Cleaning, Sentence Splitting, Scanner with Checkpoints
3.1. Cleaning
- NFC; URLs and zero-width characters removed; split at line breaks and `. ! ? ।`.
- A sentence is kept if it has 15–400 characters, ≥ 50 % Sinhala characters and ≥ 4 Sinhala tokens (same minimum as the textbook), and is not a date/time line.
- `dedup_key`: the sentence's Sinhala letters and digits only. It is used for exact deduplication per word. Near-duplicates are not removed, because one corpus is used.

In [12]:
import random, hashlib, pickle, time
import json
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

URL_PATTERN      = re.compile(r"https?://\S+|www\.\S+")
SINHALA_CHAR     = re.compile(r"[\u0D80-\u0DFF]")
NOISE_CHARS      = re.compile(r"[\u00AD\u200B\uFEFF]")
LINE_SPLIT       = re.compile(r"[\r\n]+")
SENTENCE_SPLIT   = re.compile(r"(?<=[.!?\u0964])\s*")
METADATA_PATTERN = re.compile(
    r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}"
    r"|\d{1,2}:\d{2}\s*(AM|PM)", re.IGNORECASE)
TOKEN_PATTERN    = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")
MIN_CHARS, MAX_CHARS, MIN_SINHALA_RATIO, MIN_TOKENS = 15, 400, 0.5, 4

def segment(text):
    """Document -> clean sentences."""
    text = URL_PATTERN.sub("", NOISE_CHARS.sub("", unicodedata.normalize("NFC", text or "")))
    out = []
    for line in LINE_SPLIT.split(text):
        for s in SENTENCE_SPLIT.split(re.sub(r"\s+", " ", line).strip()):
            s = s.strip()
            if not (MIN_CHARS <= len(s) <= MAX_CHARS):
                continue
            if len(SINHALA_CHAR.findall(s)) / len(s) < MIN_SINHALA_RATIO or METADATA_PATTERN.search(s):
                continue
            if len(TOKEN_PATTERN.findall(s)) < MIN_TOKENS:
                continue
            out.append(s)
    return out

def dedup_key(s):
    return re.sub(r"[^\u0D80-\u0DFF0-9]", "", match_key(s))

3.2. Word-level filters
- Keys are **spellings**, so they don't depend on how the Word cell is written.
- A sentence is dropped for a word only if **every** occurrence of the word sits inside a pattern.
- `NOISE`: the token is not the target word at all (boilerplate, abbreviations, initials, a split compound, misspellings).
- `GRAMMATICAL`: the target is used only as a function word.Use the same dict in the textbook notebook. so both sources are filtered identically.


In [13]:
S = r"[\u0D80-\u0DFF\u200C\u200D]"          # one Sinhala character (for word boundaries)
NOISE = {
    "රැස": [r"ඇස\s*පාදන\s*රැස"],                                         # website column title
    "සල්": [r"\(\s*සල්\s*\)"],                                            # Islamic honorific abbreviation
    "සී":  [rf"(?<!{S})සී\s*\.", r"විටමින්\s*සී"],                       # English letter C (initials, vitamin C)
    "මී":  [r"මී\s*ළඟ", r"මී\s*ළග"],                                     # split මීළඟ ('next')
    "දන":  [r"දන\s*(?:ගහ|ගැහ|ගැස|ගසා|ගස්ස)"],                         # misspelt දණ ගහනවා ('kneel')
}
GRAMMATICAL = {
    "කර":  [rf"කර\s*(?:ඇත|ඇති|තිබේ|තිබුණි|තිබෙන|තිබූ|ගැනීම|ගන්න|ගත්|ගෙන|දී|සිටි|සිටින|නැත)(?!{S})"],  # 'කර ඇත' = has done
    #"කලි": [r"වූ\s*කලි"],                                                 # 'වූ කලි' = when
    #"වස්": [r"නු\s*වස්"],                                                 # 'කරනු වස්' = in order to
}

def _compile(table):
    out = {}
    for spelling, pats in table.items():
        k = match_key(spelling)
        assert k in MATCH, f"Filter key '{spelling}' is not a spelling in the word list"
        out.setdefault(MATCH[k][0], []).extend(re.compile(p) for p in pats)
    return out
FILTERS = {"noise": _compile(NOISE), "grammatical": _compile(GRAMMATICAL)}

def reject_reason(word, sent, n_hits):
    """'noise' / 'grammatical' if every occurrence of the word is inside such a pattern, else None."""
    for reason, table in FILTERS.items():
        pats = table.get(word)
        if pats and sum(len(p.findall(sent)) for p in pats) >= n_hits:
            return reason
    return None

def target_info(sentence, word):
    """(target_start, target_end, n_occurrences): the first occurrence that is NOT inside a filter pattern."""
    spans = [(m.start(), m.end()) for m in TOKEN_PATTERN.finditer(sentence)
             if MATCH.get(match_key(m.group()), (None,))[0] == word]
    blocked = set()
    for table in FILTERS.values():
        for p in table.get(word, []):
            for m in p.finditer(sentence):
                for i, (start, end) in enumerate(spans):
                    if start >= m.start() and end <= m.end():
                        blocked.add(i)
    free = [span for i, span in enumerate(spans) if i not in blocked]
    start, end = (free or spans)[0]
    return start, end, len(spans)

3.3. Scanner, checkpointed extraction and outputs
- `Scanner.add`: matches whole tokens against the spellings, skips exact duplicates, applies the filters, then keeps a reservoir sample.
- Reservoir sampling (Algorithm R, Vitter 1985) gives an unbiased random sample of up to `RESERVOIR_K` sentences per word from one pass.
- `extract`: streams the corpus and saves a checkpoint every 100,000 documents; re-running resumes. A document with no `source` is counted as `unlabelled`.
- `save_outputs`: pool CSV with provenance, a summary workbook, and `meta.json`

In [14]:
class Scanner:
    """Reservoir sample (Algorithm R) of up to k sentences per word, plus counts of all distinct
    matching sentences per word / spelling / sub-corpus, and rejection counts + examples."""
    def __init__(self, name, k=RESERVOIR_K, seed=RANDOM_SEED):
        self.name, self.k, self.rng = name, k, random.Random(seed)
        self.pool, self.seen = defaultdict(list), defaultdict(set)
        self.n, self.spell_n, self.sub_n, self.rejected = Counter(), Counter(), Counter(), Counter()
        self.rejected_examples = defaultdict(list)

    def add(self, sent, h, sub_source, doc_id):
        hits, first = Counter(), {}
        for tok in TOKEN_PATTERN.findall(sent):
            m = MATCH.get(match_key(tok))
            if m:
                hits[m[0]] += 1; first.setdefault(m[0], (tok, m[1]))
        for word, n_hits in hits.items():
            if h in self.seen[word]:
                continue
            self.seen[word].add(h)
            reason = reject_reason(word, sent, n_hits)
            if reason:
                self.rejected[(word, reason)] += 1
                if len(self.rejected_examples[(word, reason)]) < 20:
                    self.rejected_examples[(word, reason)].append(sent)
                continue
            start, end, n_occ = target_info(sent, word)            # target position, recorded during the scan
            matched = sent[start:end]                              # exact text at the target position
            spelling = MATCH[match_key(matched)][1]
            self.n[word] += 1
            self.spell_n[(word, spelling)] += 1
            self.sub_n[(word, sub_source)] += 1
            row = {"No": word_no[word], "word": word, "matched": matched, "source": self.name,
                   "sub_source": sub_source, "doc_id": doc_id, "sentence": sent,
                   "target_start": start, "target_end": end, "n_occurrences": n_occ}
            pool = self.pool[word]
            if len(pool) < self.k:
                pool.append(row)
            else:
                j = self.rng.randint(1, self.n[word])
                if j <= self.k:
                    pool[j - 1] = row

def extract(run, dataset, text_key="text", sub_key="source", id_key=None, max_docs=None, every=100_000):
    """Scan the corpus ONCE; checkpoint every `every` docs; re-running resumes. One checkpoint per run name."""
    ckpt = f"{OUT}/ckpt_{run}.pkl"
    def save():
        pickle.dump({**state, "scanner": state["scanner"].__dict__}, open(ckpt, "wb"))
    if os.path.exists(ckpt):
        state = pickle.load(open(ckpt, "rb"))
        sc = Scanner.__new__(Scanner); sc.__dict__.update(state["scanner"]); state["scanner"] = sc
        print(f"Resuming {run} after {state['docs_done']:,} documents")
        dataset = dataset.skip(state["docs_done"])
    else:
        state = {"docs_done": 0, "docs": Counter(), "scanner": Scanner(run), "minutes": 0.0}
    t0 = time.time()
    for rec in tqdm(dataset, desc=run, initial=state["docs_done"]):
        if max_docs and state["docs_done"] >= max_docs:
            break
        sub = rec.get(sub_key) or "unlabelled"          # never relabel missing sources as the corpus name
        doc_id = rec.get(id_key) if id_key else state["docs_done"]
        state["docs_done"] += 1; state["docs"][sub] += 1
        for sent in segment(rec.get(text_key)):
            h = int.from_bytes(hashlib.blake2b(dedup_key(sent).encode(), digest_size=8).digest(), "big")
            state["scanner"].add(sent, h, sub, doc_id)
        if state["docs_done"] % every == 0:
            save()
    state["minutes"] += (time.time() - t0) / 60
    save()
    print(f"{run}: {state['docs_done']:,} docs, {state['minutes']:.1f} min | per sub-corpus: {dict(state['docs'])}")
    save_outputs(state["scanner"], state)
    return state["scanner"]

def save_outputs(sc, state):
    """Pool CSV (with provenance), per-word summary workbook and metadata JSON."""
    tag = f"{OUT}/{sc.name}"
    df = pd.DataFrame([r for w in TARGETS for r in sc.pool[w]],
                      columns=["No", "word", "matched", "source", "sub_source", "doc_id", "sentence",
                               "target_start", "target_end", "n_occurrences"])
    assert all(r.sentence[r.target_start:r.target_end] == r.matched for r in df.itertuples()), "bad target span"
    df.to_csv(f"{tag}_pool.csv", index=False, encoding="utf-8")
    summary = pd.DataFrame([{
        "No": word_no[w], "word": w, "matches_kept": sc.n[w], "in_pool": len(sc.pool[w]),
        "rejected_noise": sc.rejected[(w, "noise")], "rejected_grammatical": sc.rejected[(w, "grammatical")],
        "by_spelling": ", ".join(f"{s} ({c})" for (ww, s), c in sc.spell_n.most_common() if ww == w),
        "by_sub_corpus": ", ".join(f"{s} ({c})" for (ww, s), c in sc.sub_n.most_common() if ww == w),
    } for w in TARGETS])
    with pd.ExcelWriter(f"{tag}_summary.xlsx") as xw:
        summary.to_excel(xw, sheet_name="per_word", index=False)
        pd.DataFrame([{"sub_corpus": k, "documents": v} for k, v in state["docs"].items()]
                     ).to_excel(xw, sheet_name="documents", index=False)
    meta = {
        "corpus": DATASET, "revision": REVISION, "run_name": sc.name, "date": time.strftime("%Y-%m-%d"),
        "citation": "Ranasinghe et al. (2025). Sinhala Encoder-only Language Models and Evaluation. ACL 2025.",
        "word_list": os.path.basename(WORDS_XLSX), "word_sheet": WORDS_SHEET, "n_words": len(TARGETS),
        "matching": "base form only; any listed spelling = same word; whole tokens; NFC; ZWJ/ZWNJ-insensitive",
        "cleaning": {"min_chars": MIN_CHARS, "max_chars": MAX_CHARS, "min_sinhala_ratio": MIN_SINHALA_RATIO,
                     "min_tokens": MIN_TOKENS, "urls_removed": True, "date_time_lines_removed": True},
        "dedup": "exact, per word (hash of Sinhala letters + digits); no near-duplicate removal",
        "sampling": {"method": "reservoir (Algorithm R)", "k_per_word": sc.k, "seed": RANDOM_SEED},
        "noise_patterns": NOISE, "grammatical_patterns": GRAMMATICAL,
        "documents": state["docs_done"], "documents_per_sub_corpus": dict(state["docs"]),
        "scan_minutes": round(state["minutes"], 1),
        "matches_kept_total": int(summary.matches_kept.sum()),
        "rejected": {f"{w} | {r}": c for (w, r), c in sc.rejected.items()},
        "words_with_0": list(summary.loc[summary.matches_kept == 0, "word"]),
    }
    json.dump(meta, open(f"{tag}_meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"  [{sc.name}] {len(df):,} sentences in pool | words with 0: {meta['words_with_0']}")

### 4. Extract Sentences

In [ ]:
ds = load_dataset(DATASET, split="train", streaming=True, revision=REVISION)
corpus = extract("corpus15b_v5", ds)

In [16]:
# RECOVERY ONLY: use if the scan finished but the output files were not written
RUN = "corpus15b_v5"
with open(f"{OUT}/ckpt_{RUN}.pkl", "rb") as f:
    state = pickle.load(f)
sc = Scanner.__new__(Scanner)
sc.__dict__.update(state["scanner"])
state["scanner"] = sc
print(f"{state['docs_done']:,} documents in checkpoint | run name: {sc.name}")
assert sc.name == RUN, "this checkpoint belongs to another run"
save_outputs(sc, state)
corpus = sc

7,074,641 documents in checkpoint | run name: corpus15b_v5
  [corpus15b_v5] 68,648 sentences in pool | words with 0: []


### 5. Diverse sample for annotation
- Input: the 1,000-per-word random pool (`corpus15b_v5_pool.csv`).
- Near-duplicate reposts removed (same first 8 and last 4 words); sentences already in the textbook set removed.
- Equal shares per text type (web / news / SinMin / social). A type with fewer sentences gives all it has and its unused share passes to the others.


In [17]:
import os, json, random
import pandas as pd
from collections import Counter

RUN          = "corpus15b_v5"
N_PER_WORD   = 200
SEED         = 42
POOL_CSV     = f"{OUT}/{RUN}_pool.csv"
TEXTBOOK_CSV = f"{BASE}/textbooks/Ext_Final/sinhala_lit_g10_11_docai/sinhala_lit_g10_11_target_word_sentences.csv"

# sub-corpus -> text type. Web crawls = web; NSina = news; SinMin = mixed genres
# (literature, textbooks, Wikipedia, gazettes, subtitles); Facebook + Twitter = social media.
DOMAIN = {"hplt_2": "web", "CulturaX": "web", "unlabelled": "web", "Nsina": "news",
          "sinmin": "sinmin", "FacebookDecadeCorpora": "social", "semisold": "social"}

def neighbours(sentence, start, end):
    """Sinhala word just before and just after the target ('' at the sentence edge)."""
    left = TOKEN_PATTERN.findall(sentence[:start])
    right = TOKEN_PATTERN.findall(sentence[end:])
    left_word = match_key(left[-1]) if left else ""
    right_word = match_key(right[0]) if right else ""
    return left_word, right_word

def near_dup_key(sentence):
    """Same first 8 and last 4 words = the same sentence reposted with small edits."""
    tokens = [match_key(t) for t in TOKEN_PATTERN.findall(sentence)]
    return tuple(tokens[:8]), tuple(tokens[-4:])

pool = pd.read_csv(POOL_CSV).drop_duplicates(subset=["word", "sentence"])
print(f"Pool: {len(pool):,} sentences for {pool.word.nunique()} words")

# remove sentences that are also in the textbook set (kept textbook rows only)
removed_textbook = 0
if os.path.exists(TEXTBOOK_CSV):
    tb = pd.read_csv(TEXTBOOK_CSV)
    tb = tb[tb.filter_reason.fillna("") == ""]
    textbook_keys = {(w, dedup_key(s)) for w, s in zip(tb.word, tb.sentence)}
    in_textbook = [(w, dedup_key(s)) in textbook_keys for w, s in zip(pool.word, pool.sentence)]
    removed_textbook = sum(in_textbook)
    pool = pool[[not x for x in in_textbook]].copy()
else:
    print("WARNING: textbook CSV not found - overlap with the textbook not removed")
print(f"Removed {removed_textbook} sentences also in the textbook")

# the target position is already in the pool (target_start / target_end) - no need to compute it again
pool["domain"] = pool.sub_source.map(DOMAIN).fillna("web")
left_right = [neighbours(s, a, b) for s, a, b in zip(pool.sentence, pool.target_start, pool.target_end)]
pool["left_word"] = [lr[0] for lr in left_right]
pool["right_word"] = [lr[1] for lr in left_right]

rng, parts, rows = random.Random(SEED), [], []
for w in TARGETS:
    g = pool[pool.word == w].sample(frac=1, random_state=rng.randint(0, 2**31 - 1))  # random order = tie-break
    g = g[~g.sentence.map(near_dup_key).duplicated()]                                 # drop near-duplicates
    n_neardup = int((pool.word == w).sum() - len(g))
    recs = g.to_dict("records")                       # list position = random rank

    queues = {}                                       # text type -> positions of its sentences
    for i, r in enumerate(recs):
        queues.setdefault(r["domain"], []).append(i)
    used_docs, neighbour_count, spelling_count, picked = set(), Counter(), Counter(), []

    def priority(i):
        """Lower is better: unused document, then least-used neighbour words, then least-used spelling."""
        r = recs[i]
        doc_used = r["doc_id"] in used_docs
        neighbour_use = max(neighbour_count["L" + r["left_word"]], neighbour_count["R" + r["right_word"]])
        return (doc_used, neighbour_use, spelling_count[r["matched"]], i)

    while len(picked) < N_PER_WORD and any(queues.values()):
        for d in sorted(queues):                      # round-robin over text types = equal shares
            if len(picked) >= N_PER_WORD or not queues[d]:
                continue
            best = min(queues[d], key=priority)
            queues[d].remove(best)
            picked.append(best)
            r = recs[best]
            used_docs.add(r["doc_id"])
            neighbour_count["L" + r["left_word"]] += 1
            neighbour_count["R" + r["right_word"]] += 1
            spelling_count[r["matched"]] += 1

    take = pd.DataFrame([recs[i] for i in picked], columns=list(g.columns)).assign(selection="diverse")
    parts.append(take)
    rows.append({"No": word_no[w], "word": w, "available": len(g), "near_duplicates_removed": n_neardup,
                 "sampled": len(take), "short_of_target": max(0, N_PER_WORD - len(take)),
                 "distinct_documents": take.doc_id.nunique(),
                 "distinct_left_words": take.left_word.nunique(), "distinct_right_words": take.right_word.nunique(),
                 "by_domain": ", ".join(f"{k} ({v})" for k, v in take.domain.value_counts().items()),
                 "by_spelling": ", ".join(f"{k} ({v})" for k, v in take.matched.value_counts().items())})

sample = pd.concat(parts, ignore_index=True)
summary = pd.DataFrame(rows)
tag = f"{OUT}/{RUN}_{N_PER_WORD}_diverse"
cols = ["No", "word", "matched", "source", "sub_source", "domain", "doc_id", "sentence",
        "target_start", "target_end", "n_occurrences", "left_word", "right_word", "selection"]
sample[cols].to_csv(f"{tag}.csv", index=False, encoding="utf-8")
with open(f"{tag}.txt", "w", encoding="utf-8") as f:
    blocks = []
    for w in TARGETS:
        sents = [s.replace(";", ",") for s in sample.loc[sample.word == w, "sentence"]]
        blocks.append(f"s_word: {w}\nsentences: " + "; ".join(sents))
    f.write("\n\n".join(blocks))
summary.to_excel(f"{tag}_summary.xlsx", index=False)

meta_path = f"{OUT}/{RUN}_meta.json"
with open(meta_path, encoding="utf-8") as f:
    meta = json.load(f)
meta["annotation_sample_diverse"] = {
    "n_per_word": N_PER_WORD, "seed": SEED, "removed_textbook_overlap": removed_textbook,
    "textbook_csv": os.path.basename(TEXTBOOK_CSV), "domains": DOMAIN,
    "method": "near-duplicates removed (same first 8 and last 4 words); round-robin over text types (equal shares, "
              "unused share passed on); within a type: unused document first, then least-used neighbour words "
              "(left/right of target), then least-used spelling; ties in random order (seed)",
    "sentences": len(sample),
    "words_short": summary.loc[summary.short_of_target > 0, "word"].tolist()}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=1)

print(f"{len(sample):,} sentences for {sample.word.nunique()} words")
short = summary[summary.short_of_target > 0]
print("Words with fewer than 200:\n" + short[["word", "available"]].to_string(index=False) if len(short)
      else "Every word has 200.")
print(summary[["word", "sampled", "distinct_documents", "distinct_right_words", "by_domain"]].head(10).to_string(index=False))

Pool: 68,648 sentences for 70 words
Removed 1 sentences also in the textbook
14,000 sentences for 70 words
Every word has 200.
    word  sampled  distinct_documents  distinct_right_words                                      by_domain
     දිය      200                 200                   112   news (66), sinmin (66), web (65), social (3)
      දද      200                 196                   178  web (137), sinmin (50), news (12), social (1)
      නා      200                 199                   155   sinmin (72), web (71), news (48), social (9)
      ගත      200                 200                   105   news (65), sinmin (64), web (64), social (7)
පස / පස්      200                 200                   161   news (66), sinmin (65), web (65), social (4)
     මුව      200                 199                   149   sinmin (73), web (72), news (53), social (2)
වත / වත්      200                 199                   197  web (128), sinmin (44), news (19), social (9)
     තෙර      200